# Import Required Libraries
Import necessary libraries such as pandas for data manipulation, json for output, and re for regex pattern matching.

In [1]:
import pandas as pd
import json
import re
import os
import glob

In [10]:
# Load and Parse the CITES CSV File
cites_df = pd.read_csv('./CITES.csv', header=None)
# The CSV has columns: ,I,II,III
# Rename columns
cites_df.columns = ['species', 'I', 'II', 'III']

In [27]:
# Extract Species Information and Appendices
species_data = []
for index, row in cites_df.iterrows():
    for app in ['I', 'II', 'III']:
        if not pd.isna(row[app]) and row[app].strip():
            species_full = row[app].strip()
            # Parse name and note
            match = re.match(r'(.+?)\s*\((.+)\)$', species_full)
            if match:
                name = match.group(1).strip()
                note = match.group(2).strip()
            else:
                name = species_full
                note = ''
            species_data.append({
                'name': name,
                'appendix': app,
                'note': note
            })

In [28]:
# Filter Species Relevant to Vietnam
# Load Vietnam species from lib
lib_path = '../../lib'
json_files = glob.glob(os.path.join(lib_path, '*.json'))
vietnam_species = set()
for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        if isinstance(data, list):
            for item in data:
                if 'scientific_name' in item and 'value' in item['scientific_name']:
                    vietnam_species.add(item['scientific_name']['value'])
# Now, filter species_data where name in vietnam_species or genus match
filtered_species = []
for item in species_data:
    name = item['name']
    if name in vietnam_species:
        filtered_species.append(item)
    elif ' spp.' in name:
        genus = name.replace(' spp.', '')
        for vn in vietnam_species:
            if vn.startswith(genus + ' '):
                # Add for this species
                filtered_species.append({
                    'name': vn,
                    'appendix': item['appendix'],
                    'note': item['note']
                })

In [34]:
species_data

[{'name': 'I', 'appendix': 'I', 'note': ''},
 {'name': 'II', 'appendix': 'II', 'note': ''},
 {'name': 'III', 'appendix': 'III', 'note': ''},
 {'name': 'Antilocapra americana',
  'appendix': 'I',
  'note': 'Only the population of Mexico is included in Appendix I. No other population is included in the Appendices.'},
 {'name': 'Addax nasomaculatus', 'appendix': 'I', 'note': ''},
 {'name': 'Ammotragus lervia', 'appendix': 'II', 'note': ''},
 {'name': 'Antilope cervicapra', 'appendix': 'III', 'note': 'Nepal, Pakistan'},
 {'name': 'Bos gaurus {Excludes the domesticated form, which is referenced as Bos frontalis, and is not subject to the provisions of the Convention}',
  'appendix': 'I',
  'note': ''},
 {'name': 'Bos mutus {Excludes the domesticated form, which is referenced as Bos grunniens, and is not subject to the provisions of the Convention}',
  'appendix': 'I',
  'note': ''},
 {'name': 'Bos sauveli', 'appendix': 'I', 'note': ''},
 {'name': 'Boselaphus tragocamelus', 'appendix': 'III'

In [29]:
# Construct the JSON Data Structure
result = []
for item in filtered_species:
    # Parse note for geographic and codes
    note = item['note']
    geographic = note
    codes = []
    # Extract codes like A1, P2, #3
    code_pattern = r'\b(A\d+|P\d+|#\d+)\b'
    matches = re.findall(code_pattern, note)
    codes = matches
    if codes:
        geographic = re.sub(code_pattern, '', note).strip()
        if geographic.endswith('.'):
            geographic = geographic[:-1]
        combined_note = geographic + '|' + '|'.join(codes) if geographic else '|'.join(codes)
    else:
        combined_note = geographic
    laws = [{
        "name": {
            "vi": "CITES",
            "en": "CITES"
        },
        "value": item['appendix'],
        "note": combined_note
    }]
    result.append({
        "scientific_name": {
            "value": item['name'],
            "note": ""
        },
        "common_name": {
            "value": "",
            "note": ""
        },
        "kingdom_latin": "",
        "kingdom_vi": "",
        "phylum_latin": "",
        "phylum_vi": "",
        "class_latin": "",
        "class_vi": "",
        "order_latin": "",
        "order_vi": "",
        "family_latin": "",
        "family_vi": "",
        "note": "",
        "laws": laws
    })

In [ ]:
# Save the Data to a JSON File
with open('./cites.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

In [31]:
print(len(species_data))
print(len(filtered_species))
print(len(result))

1413
129
129
